# SRE agent

An incident comes in. One local model chooses the next MCP tool, reads the observations, and proposes a next step. It does not run the action.

Use the **PD Home Task** kernel. Run the setup cells once, then any of the three cases. The first model load downloads `mlx-community/Qwen3-8B-4bit` (about 4.6 GB) and runs it on the Mac GPU with MLX. Thinking is on, so each case takes about a minute or two. Override the model with `SRE_MODEL`, for example `mlx-community/Qwen3-8B-bf16` on a Mac with 24 GB or more of memory.

## Setup

### 1. Imports

In [1]:
import json
import logging

from IPython.display import Markdown, display

logging.basicConfig(level=logging.WARNING, format="%(message)s", force=True)
logging.getLogger("sre_agent").setLevel(logging.INFO)

from sre_agent.graph import build_graph, initial_state
from sre_agent.mcp.client import McpClient
from sre_agent.models import IncidentState
from sre_agent.reasoner import HuggingFaceReasoner

### 2. Load the model

This is the slow step. `tqdm` bars the file download and the weight load.

In [2]:
reasoner = HuggingFaceReasoner(thinking=True) ## Off is faster but lacks good results
print(f"Loading: {reasoner.model_id}")
reasoner.load()

Loading: mlx-community/Qwen3-8B-4bit


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

### 3. Build the graph

One orchestrator and a mocked MCP client. After PagerDuty returns the incident details, a graph node retrieves relevant local incident memory before the model chooses its next step. `run_case` shows the recalled memory, tool calls, and final proposal.

In [3]:
graph = build_graph(reasoner, McpClient())


def run_case(incident_id: str) -> IncidentState:
    print(f"\n=== {incident_id} ===", flush=True)
    reasoner.replies.clear()
    result = graph.invoke(initial_state(incident_id))
    state = result if isinstance(result, IncidentState) else IncidentState.model_validate(result)
    print(f"\n--- outcome ---\nstatus: {state.status}\nsteps: {state.step_count}")
    if state.escalate_reason:
        print(f"reason: {state.escalate_reason}")
    if state.proposal:
        print(f"\n--- proposal ---\n{state.proposal.model_dump_json(indent=2, exclude={'summary'})}")
        display(Markdown(f"### Final answer\n\n{state.proposal.summary}"))
    return state

## Case 1: bad deploy (GitLab)

`PINC2041`: `checkout-api` error rate crossed 5%. A production deploy landed twelve minutes before the alert, and its commit changes how HTTP 409 from `payments-db` is handled. The deploy history in GitLab is what explains this one. The expected proposal is a rollback of `a1b2c3d`, with approval.

In [4]:
case_1 = run_case("PINC2041")


=== PINC2041 ===


[bootstrap] loading incident PINC2041


[step 1] pagerduty.get_incident
  args:
    {
      "incident_id": "PINC2041"
    }
  returned:
    {
      "id": "PINC2041",
      "incident_number": 2041,
      "title": "checkout-api error rate above 5%",
      "status": "triggered",
      "urgency": "high",
      "service": {
        "id": "PSVCCHK",
        "summary": "checkout-api"
      },
      "created_at": "2026-09-25T00:40:00Z"
    }



[memory] no similar incidents found; continuing with an empty list


[step 2] model chose gitlab.list_deployments


...

[step 3] grafana.query_prometheus
  args:
    {
      "datasourceUid": "prom-prod",
      "expr": "sum by (service) (rate(checkout-api_errors_total{service=\"checkout-api\"} [5m]))",
      "endTime": "2026-09-25T00:40:00Z",
      "startTime": "2026-09-25T00:28:00Z",
      "stepSeconds": 60,
      "queryType": "range"
    }
  returned:
    {
      "data": {
        "result": [
          {
            "metric": {
              "service": "checkout-api"
            },
            "values": [
              [
                "error_rate",
                {
                  "current": 0.08,
                  "baseline": 0.002,
                  "rose_at": "2026-09-25T00:29:00Z"
                }
              ],
              [
                "latency_p95_ms",
                {
                  "current": 1200,
                  "baseline": 180
                }
              ],
              [
                "request_rate_rps",
                {
                  "current": 410,
       

[step 4] model proposed rollback_deploy


[finish] proposal ready for human approval



--- outcome ---
status: proposed
steps: 4

--- proposal ---
{
  "rationale": "The checkout-api error rate (8%) is 400x above its baseline (0.2%) and spiked after a deployment that changed how HTTP 409 errors from payments-db are handled. Latency_p95 is 6x higher than baseline (1200ms vs 180ms), while traffic (410 RPS) and CPU (41%) show minor deviations. The deployment (SHA a1b2c3d) occurred 12 minutes before the incident, matching the error rate's rise. No prior similar incidents match this error pattern.",
  "hypothesis": "The deployment's retry logic for HTTP 409 errors from payments-db is causing excessive retries, inflating the error rate and latency.",
  "action": "rollback_deploy",
  "proposed_action": "Rollback checkout-api deployment SHA a1b2c3d to revert the HTTP 409 retry change.",
  "needs_human_approval": true
}


### Final answer

**Issue:** checkout-api error rate (8%) and latency_p95 (1200ms) spiked after a production deployment. [grafana.query_prometheus] **Findings:** - Error rate 400x baseline [grafana.query_prometheus]. - Latency_p95 6x higher [grafana.query_prometheus]. - Deployment SHA a1b2c3d occurred 12 minutes before incident [gitlab.list_deployments]. **Fix:** Rollback deployment SHA a1b2c3d. **Verify:** Confirm error rate drops below 0.2% and latency_p95 < 180ms.

## Case 2: traffic surge (Grafana)

`PINC2052`: `search-api` p95 latency passed 1s. There has been no deploy for three days, so GitLab rules out a code change. The metrics in Grafana carry the answer: request rate is about 3x baseline, CPU is at 97%, and the service runs 4 of a possible 12 replicas. The expected proposal is to scale out, with approval.

In [5]:
case_2 = run_case("PINC2052")


=== PINC2052 ===


[bootstrap] loading incident PINC2052


[step 1] pagerduty.get_incident
  args:
    {
      "incident_id": "PINC2052"
    }
  returned:
    {
      "id": "PINC2052",
      "incident_number": 2052,
      "title": "search-api p95 latency above 1s",
      "status": "triggered",
      "urgency": "high",
      "service": {
        "id": "PSVCSRCH",
        "summary": "search-api"
      },
      "created_at": "2026-09-25T09:15:00Z"
    }



[memory] no similar incidents found; continuing with an empty list


[step 2] model chose gitlab.list_deployments


...

[step 3] grafana.query_prometheus
  args:
    {
      "datasourceUid": "prom-prod",
      "expr": "service=\"search-api\"",
      "endTime": "2026-09-25T09:15:00Z",
      "startTime": "2026-09-22T09:15:00Z",
      "stepSeconds": 3600,
      "queryType": "range"
    }
  returned:
    {
      "data": {
        "result": [
          {
            "metric": {
              "service": "search-api"
            },
            "values": [
              [
                "error_rate",
                {
                  "current": 0.004,
                  "baseline": 0.003
                }
              ],
              [
                "latency_p95_ms",
                {
                  "current": 1450,
                  "baseline": 220,
                  "rose_at": "2026-09-25T09:02:00Z"
                }
              ],
              [
                "request_rate_rps",
                {
                  "current": 3100,
                  "baseline": 950,
                  "rose_at": 

[step 4] model proposed scale_service


[finish] proposal ready for human approval



--- outcome ---
status: proposed
steps: 4

--- proposal ---
{
  "rationale": "The search-api's p95 latency (1450ms) is 6.6 times higher than its baseline (220ms), coinciding with a 230% increase in request rate (3100 RPS vs 950 baseline). CPU utilization is at 97% (baseline 45%), and replicas are at 4/12. No recent deployments to production since the traffic spike began. These metrics indicate CPU saturation from traffic growth, not a code change.",
  "hypothesis": "The search-api's replicas are saturated by traffic growth, causing CPU overload and latency spikes.",
  "action": "scale_service",
  "proposed_action": "Scale search-api from 4 to 6 replicas",
  "needs_human_approval": true
}


### Final answer

**Issue:** search-api p95 latency above 1s since 2026-09-25T09:02Z. **Findings:** - Latency_p95_ms 1450ms [grafana.query_prometheus]. - Request_rate_rps 3100 RPS [grafana.query_prometheus]. - CPU_utilization 97% [grafana.query_prometheus]. **Fix:** Scale search-api from 4 to 6 replicas. **Verify:** Check if latency_p95_ms drops below 1000ms.

## Case 3: failing dependency (local incident memory)

`PINC2063`: `orders-api` 5xx passed 2%. As soon as PagerDuty identifies the service, the memory node recalls an anonymized global pattern: an API returning 5xx because a scheduled reconciliation job exhausted its database connection pool, resolved by pausing the job. The model treats that as a prior, checks current metrics for the same signature (most 5xx are `payments-db` connection pool timeouts), and proposes pausing the reconciliation job.

In [6]:
case_3_state = run_case("PINC2063")


=== PINC2063 ===


[bootstrap] loading incident PINC2063


[step 1] pagerduty.get_incident
  args:
    {
      "incident_id": "PINC2063"
    }
  returned:
    {
      "id": "PINC2063",
      "incident_number": 2063,
      "title": "orders-api 5xx responses above 2%",
      "status": "triggered",
      "urgency": "high",
      "service": {
        "id": "PSVCORD",
        "summary": "orders-api"
      },
      "created_at": "2026-09-25T13:05:00Z"
    }



[memory] retrieved 1 similar incident(s):
[
  {
    "incident_id": "ANON-DB-POOL-001",
    "scope": "global",
    "service": "api-service",
    "summary": "An API returned elevated 5xx responses because a database connection pool was exhausted by a scheduled reconciliation job.",
    "resolution": "Pause the reconciliation job, then verify database connection availability and 5xx responses return to baseline.",
    "score": 0.4126
  }
]


[step 2] model chose grafana.query_prometheus


[step 2] grafana.query_prometheus
  args:
    {
      "datasourceUid": "prom-prod",
      "expr": "sum by (service) (rate(http_requests_total{job=\"orders-api\", status!=\"200\"} [5m]))",
      "endTime": "2026-09-25T13:05:00Z",
      "startTime": "2026-09-25T12:00:00Z",
      "stepSeconds": 300
    }
  returned:
    {
      "data": {
        "result": [
          {
            "metric": {
              "service": "orders-api"
            },
            "values": [
              [
                "error_rate",
                {
                  "current": 0.031,
                  "baseline": 0.001,
                  "rose_at": "2026-09-25T13:01:00Z"
                }
              ],
              [
                "latency_p95_ms",
                {
                  "current": 2900,
                  "baseline": 240
                }
              ],
              [
                "request_rate_rps",
                {
                  "current": 520,
                  "baseline": 

[step 3] model proposed pause_job


[finish] proposal ready for human approval



--- outcome ---
status: proposed
steps: 3

--- proposal ---
{
  "rationale": "The orders-api has a 3.1% error rate (baseline 0.1%), p95 latency of 2900ms (baseline 240ms), and 94% of its 5xx errors originate from payments-db connection pool timeouts. This matches the pattern of the previous incident where a reconciliation job exhausted the database connection pool.",
  "hypothesis": "The orders-api is experiencing 5xx errors due to the payments-db connection pool being exhausted by a scheduled reconciliation job.",
  "action": "pause_job",
  "proposed_action": "Pause the payments-db reconciliation job to relieve the connection pool and verify if 5xx errors subside.",
  "needs_human_approval": true
}


### Final answer

**Issue:** orders-api has 3.1% 5xx errors since 2026-09-25T13:01Z, with 94% linked to payments-db connection pool timeouts. **Findings:** - Error rate 3.1% (baseline 0.1%) [grafana.query_prometheus]. - 94% of 5xx errors from payments-db [grafana.query_prometheus]. **Fix:** Pause payments-db reconciliation job. **Verify:** Monitor 5xx error rate and payments-db connection pool metrics.